# CAI 5607 — Homework 1
## Question 5: Tiny Character-Level Neural Language Model

**Goal:** Predict the next character from the previous eight characters.

Keep `input.txt` in the same folder as this notebook. Complete every cell marked
`TODO`, preserve the validation checks, and run the notebook from beginning to end
before submission.


In [ ]:
# Imports and configuration
from pathlib import Path
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
CONTEXT_LENGTH = 8
EMBEDDING_DIM = 32
HIDDEN_DIM = 128
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
TRAINING_STEPS = 1500
EVAL_INTERVAL = 100

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch version:", torch.__version__)


## A. Prepare the Text Data — 8 Points

In [ ]:
# A1. Load the corpus.
# input.txt is the required filename. input.tx is accepted only for compatibility.
candidate_paths = [Path("input.txt"), Path("input.tx")]
DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Place input.txt in the same folder as this notebook."
    )

text = DATA_PATH.read_text(encoding="utf-8")
print("Corpus:", DATA_PATH)
print("Number of characters:", len(text))
print("First 200 characters:")
print(repr(text[:200]))


In [ ]:
# A2. Build the character vocabulary.
# TODO: Replace the four None values.

chars = None       # Hint: sorted list of unique characters in text
vocab_size = None
stoi = None        # character -> integer ID
itos = None        # integer ID -> character

assert chars is not None, "TODO: build chars."
assert vocab_size == len(chars)
assert len(stoi) == vocab_size
assert len(itos) == vocab_size

print("Vocabulary size:", vocab_size)
print("Vocabulary:", repr("".join(chars)))


In [ ]:
# A3. Implement encode() and decode().

def encode(string):
    """Convert a string into a list of character IDs."""
    # TODO
    raise NotImplementedError


def decode(token_ids):
    """Convert character IDs back into a string."""
    # TODO
    raise NotImplementedError


assert decode(encode("hello")) == "hello"
print("Encoding/decoding check passed.")


In [ ]:
# A4. Create a contiguous 90/10 split.
data = torch.tensor(encode(text), dtype=torch.long)

split_index = int(0.9 * len(data))
train_data = data[:split_index]
val_data = data[split_index:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))


In [ ]:
# A5. Create context-target examples.

def make_examples(sequence, context_length=CONTEXT_LENGTH):
    """Return inputs [N, context_length] and targets [N]."""
    contexts = []
    targets = []

    # TODO: For each valid position, append:
    # sequence[i : i + context_length] to contexts
    # sequence[i + context_length] to targets

    raise NotImplementedError


train_x, train_y = make_examples(train_data)
val_x, val_y = make_examples(val_data)

assert train_x.ndim == 2 and train_x.shape[1] == CONTEXT_LENGTH
assert train_y.ndim == 1 and len(train_x) == len(train_y)
assert val_x.ndim == 2 and val_x.shape[1] == CONTEXT_LENGTH
assert val_y.ndim == 1 and len(val_x) == len(val_y)

print("Training inputs:", tuple(train_x.shape))
print("Training targets:", tuple(train_y.shape))


In [ ]:
# Display at least three decoded examples.
for i in range(3):
    context = decode(train_x[i].tolist())
    target = decode([train_y[i].item()])
    print(f"{i + 1}. {context!r} -> {target!r}")


## B. Build a Bigram Baseline — 8 Points

In [ ]:
# B1. Count consecutive character transitions.
counts = torch.zeros((vocab_size, vocab_size), dtype=torch.float32)

# TODO: For every consecutive pair in train_data, add 1 to counts[current, next].

assert counts.shape == (vocab_size, vocab_size)
print("Counted transitions:", int(counts.sum().item()))


In [ ]:
# B2. Apply add-one smoothing.
# TODO: Complete both lines.

smoothed_counts = None
bigram_probs = None

assert bigram_probs is not None
assert torch.allclose(
    bigram_probs.sum(dim=1),
    torch.ones(vocab_size),
    atol=1e-5,
)
print("Each probability row sums to 1.")


In [ ]:
# B3. Evaluate the bigram model.

def bigram_nll(sequence, probabilities):
    """Return mean token-level negative log-likelihood."""
    # TODO:
    # 1. Look up p(next | current) for each consecutive pair.
    # 2. Take the negative natural log.
    # 3. Return the mean.
    raise NotImplementedError


train_bigram_nll = bigram_nll(train_data, bigram_probs)
val_bigram_nll = bigram_nll(val_data, bigram_probs)
val_bigram_ppl = math.exp(val_bigram_nll)

print(f"Training NLL:   {train_bigram_nll:.4f}")
print(f"Validation NLL: {val_bigram_nll:.4f}")
print(f"Validation PPL: {val_bigram_ppl:.4f}")


In [ ]:
# B4. Generate at least 200 characters.

def generate_bigram(start_id, probabilities, length=200, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    generated = [int(start_id)]

    # TODO:
    # Repeat length times:
    # 1. Select the probability row for the latest character.
    # 2. Apply temperature safely.
    # 3. Sample the next ID with torch.multinomial.
    # 4. Append the next ID.

    raise NotImplementedError


start_id = int(train_data[0])
bigram_ids = generate_bigram(
    start_id, bigram_probs, length=200, temperature=1.0
)
print(decode(bigram_ids))


## C. Build the Neural Language Model — 12 Points

In [ ]:
class CharacterLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_length=CONTEXT_LENGTH,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
    ):
        super().__init__()
        self.context_length = context_length
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden = nn.Linear(context_length * embedding_dim, hidden_dim)
        self.activation = nn.GELU()
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids):
        # token_ids: [B, 8]

        # TODO 1: Embedding output [B, 8, 32]
        embeddings = None

        # TODO 2: Flatten to [B, 256]
        flattened = None

        # TODO 3: Linear + GELU -> [B, 128]
        hidden_features = None

        # TODO 4: Output logits -> [B, vocab_size]
        logits = None

        return logits


model = CharacterLanguageModel(vocab_size).to(device)
print(model)


In [ ]:
# C2. Confirm output shape and report trainable parameters.
sample_inputs = train_x[:BATCH_SIZE].to(device)
sample_logits = model(sample_inputs)

assert sample_logits.shape == (len(sample_inputs), vocab_size)

num_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Input shape:", tuple(sample_inputs.shape))
print("Logit shape:", tuple(sample_logits.shape))
print("Trainable parameters:", num_parameters)


**Brief response:** Why should the model return logits rather than apply softmax
before `CrossEntropyLoss`?

TODO


## D. Train and Evaluate — 10 Points

In [ ]:
# Create data loaders.
train_loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(val_x, val_y),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


In [ ]:
@torch.no_grad()
def evaluate_model(model, data_loader, loss_function, device):
    model.eval()
    total_loss = 0.0
    total_examples = 0

    for inputs, targets in data_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits = model(inputs)
        loss = loss_function(logits, targets)

        batch_size = inputs.shape[0]
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


In [ ]:
# D1-D3. Configure and train the model.
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

train_losses = []
val_losses = []
recorded_steps = []

train_iterator = iter(train_loader)
start_time = time.perf_counter()

for step in range(1, TRAINING_STEPS + 1):
    try:
        inputs, targets = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        inputs, targets = next(train_iterator)

    inputs = inputs.to(device)
    targets = targets.to(device)

    # TODO: Complete the five standard training operations:
    # optimizer.zero_grad()
    # logits = ...
    # loss = ...
    # loss.backward()
    # optimizer.step()
    raise NotImplementedError

    if step == 1 or step % EVAL_INTERVAL == 0:
        # Use the current mini-batch loss for the training curve.
        model.eval()
        val_loss = evaluate_model(
            model, val_loader, loss_function, device
        )
        model.train()

        recorded_steps.append(step)
        train_losses.append(float(loss.item()))
        val_losses.append(val_loss)

        print(
            f"Step {step:4d} | "
            f"train batch NLL {loss.item():.4f} | "
            f"validation NLL {val_loss:.4f}"
        )

training_time = time.perf_counter() - start_time
print(f"Training time: {training_time:.2f} seconds")


In [ ]:
# D4. Plot training and validation loss.
plt.figure(figsize=(8, 5))
plt.plot(recorded_steps, train_losses, label="Training batch NLL")
plt.plot(recorded_steps, val_losses, label="Validation NLL")
plt.xlabel("Training step")
plt.ylabel("Negative log-likelihood")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# D5. Report final results.
final_train_nll = evaluate_model(
    model, train_loader, loss_function, device
)
final_val_nll = evaluate_model(
    model, val_loader, loss_function, device
)
final_val_ppl = math.exp(final_val_nll)

print(f"Bigram validation NLL: {val_bigram_nll:.4f}")
print(f"Bigram validation PPL: {val_bigram_ppl:.4f}")
print(f"Neural training NLL:   {final_train_nll:.4f}")
print(f"Neural validation NLL: {final_val_nll:.4f}")
print(f"Neural validation PPL: {final_val_ppl:.4f}")
print(f"Device:                {device}")
print(f"Training time:         {training_time:.2f} seconds")


### Evaluation

1. Which model performs better?
2. Why can the neural model use more context?
3. Do the curves suggest underfitting, a reasonable fit, or overfitting?

**Response:** TODO


## E. Generate and Analyze Text — 7 Points

In [ ]:
@torch.no_grad()
def generate_neural(
    model,
    initial_context,
    length,
    temperature,
    device,
):
    if len(initial_context) != CONTEXT_LENGTH:
        raise ValueError(
            f"initial_context must contain {CONTEXT_LENGTH} characters"
        )
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    model.eval()
    generated_ids = encode(initial_context)

    for _ in range(length):
        # TODO:
        # 1. Keep the latest eight IDs.
        # 2. Create an input tensor [1, 8].
        # 3. Run the model to obtain logits.
        # 4. Divide logits by temperature.
        # 5. Convert logits to probabilities.
        # 6. Sample one next-token ID.
        # 7. Append it to generated_ids.
        raise NotImplementedError

    return decode(generated_ids)


In [ ]:
# Generate at least 300 new characters at each temperature.
initial_context = decode(train_data[:CONTEXT_LENGTH].tolist())

for temperature in [0.7, 1.0, 1.3]:
    print("=" * 80)
    print("Temperature:", temperature)
    generated = generate_neural(
        model=model,
        initial_context=initial_context,
        length=300,
        temperature=temperature,
        device=device,
    )
    print(generated)


### Generation Analysis — About 150 Words

Compare the bigram and neural outputs, the effects of temperature, local patterns and
repetition, and why this model is not a modern LLM.

**Response:** TODO


## Reproducibility and AI-Tool Disclosure

**Reproducibility**

- Random seed:
- Python version:
- PyTorch version:
- Device:
- Training steps:
- Training time:
- Hyperparameter changes:

**AI-tool disclosure**

- Tool/model:
- Purpose:
- Material incorporated:
- What you verified or changed:
